# モデルが自信満々に間違えた事例を調べる (エラー分析)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_classification/error_analysis.ipynb)

NLP クックブック 14-16 ／ テキスト分類

精度の数字だけでなく、どんな事例を間違えているのかを見たい。アノテーション誤りの発見にもつながる。

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline

cats = ["rec.sport.baseball", "rec.sport.hockey", "sci.space", "sci.med", "comp.graphics"]
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
pipe = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True), LogisticRegression(C=10, max_iter=1000))

# 交差検証で、すべての事例について「その事例を学習に使っていないモデル」の予測確率を得る
proba = cross_val_predict(pipe, data.data, data.target, cv=5, method="predict_proba", n_jobs=-1)
df = pd.DataFrame({"text": [" ".join(t.split())[:110] for t in data.data], "gold": data.target, "pred": proba.argmax(axis=1),
                   "p_gold": proba[np.arange(len(proba)), data.target], "p_pred": proba.max(axis=1)})
df["gold"], df["pred"] = df["gold"].map(lambda i: cats[i]), df["pred"].map(lambda i: cats[i])
print("正解率:", round((df.gold == df.pred).mean(), 3))

正解率: 0.89


In [2]:
pd.set_option("display.max_colwidth", 120)
errors = df[df.gold != df.pred].sort_values("p_gold")   # 正解ラベルの確率が低い順 = 自信満々に間違えた順
errors.head(8)

,text,gold,pred,p_gold,p_pred
1533,Thanks Ron and Peter for some very nice maps. I have an advice though. You wrote that the maps were reduced to,comp.graphics,rec.sport.baseball,0.027185,0.925189
927,Archive-name: typing-injury-faq/keyboards Version: $Revision: 5.11 $ $Date: 1993/04/13 01:20:43 $ ------------,sci.med,rec.sport.baseball,0.035289,0.865495
1092,I am posting the following for my brother. Please post your replies or send him email to his address at the en,sci.med,rec.sport.baseball,0.041074,0.853124
136,It's only test message.,sci.space,comp.graphics,0.041089,0.632629
2313,(FRANK MICH even show At least your parents didn't move you Idaho. The only things that get any coverage there,sci.space,rec.sport.hockey,0.048292,0.821181
495,There are several star map programs available. Your job is to choose that you like. Try anonymous-FTP from: ft,comp.graphics,rec.sport.baseball,0.054647,0.922496
2270,If anybody is having problems following the thread be sure to ask the origonal poster to rectify your misunder,sci.space,sci.med,0.055922,0.508732
979,Vielen dank Desperate Fan,sci.space,rec.sport.hockey,0.062849,0.788373


In [3]:
print("空に近い文書の割合 (誤り / 全体):", round((errors.text.str.len() < 20).mean(), 2), "/", round((df.text.str.len() < 20).mean(), 2))
pd.crosstab(errors.gold, errors.pred)

空に近い文書の割合 (誤り / 全体): 0.21 / 0.03


pred,comp.graphics,rec.sport.baseball,rec.sport.hockey,sci.med,sci.space
gold,,,,,
comp.graphics,0,18,31,19,9
rec.sport.baseball,10,0,27,15,4
rec.sport.hockey,9,9,0,4,27
sci.med,19,20,29,0,1
sci.space,11,9,49,7,0


- 上位には「本文が空」「引用だけ」「ラベルのほうが怪しい」事例が並ぶ。モデルの改善より先にデータの掃除が必要だと分かることが多い。
- この「交差検証の予測確率でラベル誤りを探す」考え方を体系化したのが Confident Learning (`cleanlab`)。